In [ ]:
# --- Dengue Data Acquisition Script (Historical 2019) ---

import requests
import pandas as pd
import geopandas as gpd
from datetime import date, timedelta
import time
import os
import glob
from tqdm import tqdm

# --- Configuration ---
DENGUE_START_DATE = date(2019, 5, 1)
DENGUE_END_DATE = date(2019, 7, 31)
LAG_MONTHS = 2
WEATHER_START_DATE = date(DENGUE_START_DATE.year, DENGUE_START_DATE.month - LAG_MONTHS, DENGUE_START_DATE.day)
WEATHER_END_DATE = date(DENGUE_END_DATE.year, DENGUE_END_DATE.month - LAG_MONTHS, DENGUE_END_DATE.day)

# --- Folders and Filenames ---
HISTORICAL_DENGUE_FOLDER = "historical_dengue_csvs"
POPULATION_CSV = "population_subzone_2020.csv"
BOUNDARY_GEOJSON = "subzone_boundaries.geojson"
OUTPUT_DIR = "downloaded_data_2019"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- API Endpoints & Helper Function ---
TEMP_API_URL = "https://api.data.gov.sg/v1/environment/air-temperature"
RAIN_API_URL = "https://api.data.gov.sg/v1/environment/rainfall"

# --- Helper Function for API Calls (Collects Metadata Iteratively) ---

def fetch_weather_data_comprehensive(api_url, start_date, end_date, data_type):
    """
    Fetches weather data day by day, collects metadata encountered,
    and returns readings and a comprehensive metadata dictionary.
    
    This version loops through ALL 'items' (all timestamps) per day.
    """
    print(f"Fetching {data_type} data & metadata from {start_date} to {end_date}...")
    all_reading_data = []
    comprehensive_metadata = {}
    current_date = start_date
    delta = timedelta(days=1)
    num_days = (end_date - start_date).days + 1
    pbar = tqdm(total=num_days, desc=f"Downloading {data_type}")

    while current_date <= end_date:
        date_str = current_date.strftime("%Y-%m-%d")
        params = {'date': date_str}
        try:
            response = requests.get(api_url, params=params)
            response.raise_for_status()
            data = response.json()

            # 1. Update Comprehensive Metadata (from daily response)
            if 'metadata' in data and 'stations' in data['metadata']:
                for station in data['metadata']['stations']:
                    station_id = station.get('id')
                    location = station.get('location')
                    if station_id and station_id not in comprehensive_metadata:
                         if location and 'latitude' in location and 'longitude' in location:
                            comprehensive_metadata[station_id] = {
                                'latitude': location['latitude'],
                                'longitude': location['longitude'],
                                'name': station.get('name', '')
                            }

            # 2. Process readings for the day
            # The API returns a list of 'items'. Each item is a timestamp.
            if data.get('items'):
                for item in data['items']: # Loop through all timestamps
                    api_timestamp = item.get('timestamp')
                    if item.get('readings'):
                        for reading in item['readings']: # Loop through all stations for that timestamp
                            station_id = reading.get('station_id')
                            value = reading.get('value')
                            if station_id and value is not None:
                                all_reading_data.append({
                                    'station_id': station_id,
                                    'value': value,
                                    'timestamp': api_timestamp,
                                    'date_retrieved': date_str
                                })

            pbar.update(1)
            time.sleep(0.2)

        except requests.exceptions.RequestException as e:
            print(f"\nError fetching {data_type} data for {date_str}: {e}")
        except Exception as e:
            print(f"\nError processing {data_type} data for {date_str}: {e}")

        current_date += delta
    pbar.close()

    if not all_reading_data:
        print(f"Warning: No {data_type} readings were successfully retrieved.")
        return None, comprehensive_metadata if comprehensive_metadata else None

    readings_df = pd.DataFrame(all_reading_data)
    print(f"Successfully fetched {len(readings_df)} {data_type} reading records.")
    print(f"Collected metadata for {len(comprehensive_metadata)} unique stations.")

    # 3. Add coordinates to readings DataFrame
    if comprehensive_metadata:
        metadata_df = pd.DataFrame.from_dict(comprehensive_metadata, orient='index')
        metadata_df = metadata_df.reset_index().rename(columns={'index': 'station_id'})
        readings_with_coords_df = pd.merge(readings_df, metadata_df, on='station_id', how='left')

        missing_coords = readings_with_coords_df['latitude'].isna().sum()
        if missing_coords > 0:
            missing_ids = readings_with_coords_df[readings_with_coords_df['latitude'].isna()]['station_id'].unique()
            print(f"WARNING: Could not find coordinates for {missing_coords} readings ({len(missing_ids)} station IDs).")

        print(f"Final {data_type} DataFrame has {len(readings_with_coords_df)} records after merging coordinates.")
        return readings_with_coords_df, comprehensive_metadata
    else:
        print("Warning: No metadata was collected. Returning readings without coordinates.")
        return readings_df, None

In [3]:
print("\n--- Fetching Historical Weather Readings & Metadata ---")
# 1. Fetch Temperature Data (Lagged Period - 2019)
temp_df, temp_meta = fetch_weather_data_comprehensive(TEMP_API_URL, WEATHER_START_DATE, WEATHER_END_DATE, "Temperature")
if temp_df is not None:
    temp_filepath = os.path.join(OUTPUT_DIR, f"temperature_{WEATHER_START_DATE}_to_{WEATHER_END_DATE}_with_coords.csv")
    temp_df.to_csv(temp_filepath, index=False)
    print(f"Temperature data saved to {temp_filepath}")
# Save combined metadata
if temp_meta:
     pd.DataFrame.from_dict(temp_meta, orient='index').to_csv(os.path.join(OUTPUT_DIR, "temp_station_metadata_collected.csv"))


# 2. Fetch Rainfall Data (Lagged Period - 2019)
rain_df, rain_meta = fetch_weather_data_comprehensive(RAIN_API_URL, WEATHER_START_DATE, WEATHER_END_DATE, "Rainfall")
if rain_df is not None:
    rain_filepath = os.path.join(OUTPUT_DIR, f"rainfall_{WEATHER_START_DATE}_to_{WEATHER_END_DATE}_with_coords.csv")
    rain_df.to_csv(rain_filepath, index=False)
    print(f"Rainfall data saved to {rain_filepath}")
if rain_meta:
     pd.DataFrame.from_dict(rain_meta, orient='index').to_csv(os.path.join(OUTPUT_DIR, "rain_station_metadata_collected.csv"))


--- Fetching Historical Weather Readings & Metadata ---
Fetching Temperature data & metadata from 2019-03-01 to 2019-05-31...


Successfully fetched 2025041 Temperature reading records.
Collected metadata for 17 unique stations.
Final Temperature DataFrame has 2025041 records after merging coordinates.
Temperature data saved to downloaded_data_2019\temperature_2019-03-01_to_2019-05-31_with_coords.csv
Fetching Rainfall data & metadata from 2019-03-01 to 2019-05-31...


Successfully fetched 1221169 Rainfall reading records.
Collected metadata for 52 unique stations.
Final Rainfall DataFrame has 1221169 records after merging coordinates.
Rainfall data saved to downloaded_data_2019\rainfall_2019-03-01_to_2019-05-31_with_coords.csv


In [ ]:
# --- 3. Process Historical Dengue Cases ---
print(f"\nProcessing historical dengue CSVs from folder: {HISTORICAL_DENGUE_FOLDER}")
all_dengue_files = glob.glob(os.path.join(HISTORICAL_DENGUE_FOLDER, "*.csv"))
if not all_dengue_files:
    print(f"ERROR: No CSV files found in '{HISTORICAL_DENGUE_FOLDER}'.")
else:
    print(f"Found {len(all_dengue_files)} historical dengue CSV files.")
    li = []
    col_names = ['Num_Cases_Loc', 'Street_Address', 'Latitude', 'Longitude',
                 'Cluster_Num', 'Recent_Cases_Cluster', 'Total_Cases_Cluster',
                 'Date_YYMMDD', 'Month_Num']

    for filename in tqdm(all_dengue_files, desc="Reading historical CSVs"):
        try:
            df = pd.read_csv(filename, header=0, names=col_names, on_bad_lines='skip')
            li.append(df)
        except Exception as e:
            print(f"\nWarning: Could not read or process {filename}: {e}")

    if li:
        dengue_hist_df = pd.concat(li, axis=0, ignore_index=True)
        print(f"Successfully combined {len(dengue_hist_df)} records.")
        try:
            dengue_hist_df['Date_YYMMDD'] = dengue_hist_df['Date_YYMMDD'].astype(str).str.replace(r'\D+', '', regex=True)
            dengue_hist_df['Full_Date_Str'] = '20' + dengue_hist_df['Date_YYMMDD'].str.zfill(6)
            dengue_hist_df['date'] = pd.to_datetime(dengue_hist_df['Full_Date_Str'], format='%Y%m%d', errors='coerce')
            dengue_hist_df.dropna(subset=['date', 'Latitude', 'Longitude'], inplace=True)
            dengue_hist_df['Latitude'] = pd.to_numeric(dengue_hist_df['Latitude'], errors='coerce')
            dengue_hist_df['Longitude'] = pd.to_numeric(dengue_hist_df['Longitude'], errors='coerce')
            dengue_hist_df.dropna(subset=['Latitude', 'Longitude'], inplace=True)

            dengue_2019_outbreak = dengue_hist_df[
                (dengue_hist_df['date'].dt.date >= DENGUE_START_DATE) &
                (dengue_hist_df['date'].dt.date <= DENGUE_END_DATE)
            ].copy()

            if not dengue_2019_outbreak.empty:
                print(f"Filtered to {len(dengue_2019_outbreak)} records for {DENGUE_START_DATE} to {DENGUE_END_DATE}.")
                
                # Prepare for Shapefile output
                dengue_to_export = dengue_2019_outbreak[['date', 'Latitude', 'Longitude']].copy()
                dengue_to_export['date_str'] = dengue_to_export['date'].dt.strftime('%Y-%m-%d') # Shapefiles prefer string dates
                dengue_to_export = dengue_to_export.drop(columns=['date']) # Drop datetime object
                
                geometry = gpd.points_from_xy(dengue_to_export.Longitude, dengue_to_export.Latitude)
                dengue_gdf = gpd.GeoDataFrame(dengue_to_export, geometry=geometry, crs="EPSG:4326")
                dengue_gdf_svy21 = dengue_gdf.to_crs("EPSG:3414")

                # Save as Shapefile, geopackage generated by Geopandas doesn't seem to work well with Arcpy for whatever reason
                dengue_filepath_shp = os.path.join(OUTPUT_DIR, f"dengue_points_{DENGUE_START_DATE}_to_{DENGUE_END_DATE}.shp")
                dengue_gdf_svy21.to_file(dengue_filepath_shp, driver="ESRI Shapefile")
                print(f"Filtered historical dengue points saved to {dengue_filepath_shp}")
            else:
                print(f"Warning: No historical dengue cases found for {DENGUE_START_DATE} to {DENGUE_END_DATE}.")
        except Exception as e:
            print(f"Error processing combined dengue data: {e}")
    else:
        print("Warning: No historical dengue data combined.")


Processing historical dengue CSVs from folder: historical_dengue_csvs
Found 256 historical dengue CSV files.


Reading historical CSVs: 100%|██████████| 256/256 [00:00<00:00, 1318.10it/s]


Successfully combined 56720 records.
Filtered to 6515 records for 2019-05-01 to 2019-07-31.
Filtered historical dengue points saved to downloaded_data_2019\dengue_points_2019-05-01_to_2019-07-31.shp


In [ ]:
# 4. Process Population Data (2020 Data)
print(f"\nProcessing manually downloaded population data: {POPULATION_CSV}")
try:
    # Read the CSV, specifying that the first row (index 0) is the header
    pop_df = pd.read_csv(POPULATION_CSV, header=0)

    # Define the expected column names
    subzone_col_name = "Number" # Column containing subzone names
    total_pop_col_name = "Total_Total" # Column containing total population

    print(f"Reading population data using columns: '{subzone_col_name}' and '{total_pop_col_name}'")

    # Check if the expected columns exist
    if subzone_col_name in pop_df.columns and total_pop_col_name in pop_df.columns:
        # Select only the relevant columns
        pop_df_filtered = pop_df[[subzone_col_name, total_pop_col_name]].copy()

        # Clean up subzone names (remove potential extra spaces)
        pop_df_filtered[subzone_col_name] = pop_df_filtered[subzone_col_name].astype(str).str.strip()

        # Filter out summary rows (like 'Total' or rows containing '- Total')
        # This assumes subzone names don't typically contain the word "Total"
        pop_df_filtered = pop_df_filtered[~pop_df_filtered[subzone_col_name].str.contains("Total", case=False, na=False)]

        # Rename columns to standard names for consistency
        pop_df_filtered.rename(columns={subzone_col_name: 'Subzone_Name', total_pop_col_name: 'Population_2020'}, inplace=True)

        # Ensure Population column is numeric and handle missing values
        pop_df_filtered['Population_2020'] = pd.to_numeric(pop_df_filtered['Population_2020'], errors='coerce')
        pop_df_filtered.dropna(subset=['Population_2020', 'Subzone_Name'], inplace=True) # Drop rows if population or name is missing
        pop_df_filtered['Population_2020'] = pop_df_filtered['Population_2020'].astype(int)

        # Save the processed data
        pop_filepath = os.path.join(OUTPUT_DIR, "population_subzone_processed.csv")
        pop_df_filtered.to_csv(pop_filepath, index=False)
        print(f"Processed population data saved to {pop_filepath}")
        print(f"Kept {len(pop_df_filtered)} rows after filtering summary totals.")

    else:
         print(f"ERROR: Expected columns '{subzone_col_name}' and/or '{total_pop_col_name}' not found in {POPULATION_CSV}. Please verify column names in the file.")

except FileNotFoundError:
    print(f"ERROR: Manual download file '{POPULATION_CSV}' not found.")
except Exception as e:
    print(f"Error processing population CSV: {e}")


Processing manually downloaded population data: population_subzone_2020.csv
Reading population data using columns: 'Number' and 'Total_Total'
Processed population data saved to downloaded_data_2019\population_subzone_processed.csv
Kept 286 rows after filtering summary totals.


In [ ]:
# 5. Process Boundary Data (GeoJSON)
print(f"\nProcessing boundary GeoJSON: {BOUNDARY_GEOJSON}")
try:
    boundaries_gdf = gpd.read_file(BOUNDARY_GEOJSON)
    subzone_name_field_geojson = 'SUBZONE_NAME'

    if subzone_name_field_geojson in boundaries_gdf.columns:
        # Keep only the essential columns and rename
        # Clean the name just in case
        boundaries_gdf_clean = boundaries_gdf[[subzone_name_field_geojson, 'geometry']].copy()
        boundaries_gdf_clean.rename(columns={subzone_name_field_geojson: 'Subzone_Name'}, inplace=True) # Ensure consistent naming
        boundaries_gdf_clean['Subzone_Name'] = boundaries_gdf_clean['Subzone_Name'].astype(str).str.strip()

        # Project to SVY21
        boundaries_gdf_svy21 = boundaries_gdf_clean.to_crs("EPSG:3414")

        # Save as GeoPackage
        boundary_filepath_gpkg = os.path.join(OUTPUT_DIR, "subzone_boundaries_svy21.gpkg") # Consistent output name
        boundaries_gdf_svy21.to_file(boundary_filepath_gpkg, driver="GPKG")
        print(f"Processed and projected boundary data saved to {boundary_filepath_gpkg}")
    else:
        print(f"ERROR: Could not find the expected subzone name field ('{subzone_name_field_geojson}') in the GeoJSON file {BOUNDARY_GEOJSON}.")
        print("Available fields:", boundaries_gdf.columns.tolist())

except FileNotFoundError:
    print(f"ERROR: Manual download file '{BOUNDARY_GEOJSON}' not found.")
except Exception as e:
    print(f"Error processing boundary GeoJSON: {e}")

print("\n--- Data Acquisition Script Finished ---")


Processing boundary GeoJSON: subzone_boundaries.geojson
Processed and projected boundary data saved to downloaded_data_2019\subzone_boundaries_svy21.gpkg

--- Data Acquisition Script Finished ---
